# Stage 4 — Agreement and faithfulness (the payoff)

**Capstone: Evaluating the Faithfulness of Attribution-Based Interpretability Tooling in GPT-2**

This notebook answers the research questions. It brings both methods together in one place and produces the actual results:

- **RQ1 — Agreement:** do the two methods rank the same heads as important, and do they recover the known IOI circuit? (rank correlation, top-k overlap, precision/recall)
- **RQ2 — Faithfulness:** when we switch off the heads each method calls important, does the behaviour actually fall away? (necessity and sufficiency curves)
- **RQ3 — Efficiency:** how much faster is attribution patching?

It is self-contained: it recomputes both importance matrices and the ablation harness, so you can run it on its own.

*Runs in Google Colab (T4 GPU). Tools: TransformerLens; methods from Wang et al. (2023) and Nanda (2023); statistics via SciPy.*

## 1. Setup, data, metric, baselines (as before)

In [ ]:
!pip -q install transformer_lens

In [ ]:
import torch, itertools, random, time, functools
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import spearmanr
from transformer_lens import HookedTransformer, utils

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)
n_layers, n_heads = model.cfg.n_layers, model.cfg.n_heads

template = "When{A} and{B} went to the shop,{S} gave a drink to"
candidate_names = [" Mary", " John", " Tom", " James", " Anna", " Kate",
                   " Mark", " Paul", " Alice", " Sarah", " David", " Emma"]
names = [n for n in candidate_names if model.to_tokens(n, prepend_bos=False).shape[1] == 1]
random.seed(0)
pairs = [(a, b) for a, b in itertools.permutations(names, 2)]
random.shuffle(pairs); pairs = pairs[:15]
clean_prompts     = [template.format(A=a, B=b, S=b) for a, b in pairs]
corrupted_prompts = [template.format(A=a, B=b, S=a) for a, b in pairs]
clean_tokens     = model.to_tokens(clean_prompts)
corrupted_tokens = model.to_tokens(corrupted_prompts)
N = len(pairs)
rows = torch.arange(N, device=device)
io_tokens = torch.tensor([model.to_single_token(a) for a, b in pairs], device=device)
s_tokens  = torch.tensor([model.to_single_token(b) for a, b in pairs], device=device)

def logit_diff_tensor(logits):
    final = logits[:, -1, :]
    return (final[rows, io_tokens] - final[rows, s_tokens]).mean()
def logit_diff(logits):
    return logit_diff_tensor(logits).item()

with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    CLEAN_LD = logit_diff(clean_logits)
    CORRUPT_LD = logit_diff(model(corrupted_tokens))
denom = CLEAN_LD - CORRUPT_LD
print(f"Clean {CLEAN_LD:+.3f} | Corrupted {CORRUPT_LD:+.3f} | scale {denom:.3f}")

## 2. Compute both importance matrices
`patch_imp` = activation patching (Stage 2, exact, slow). `attr_imp` = attribution patching (Stage 3, estimate, fast). We also time each — that is the RQ3 result.

In [ ]:
# --- Activation patching (exact) ---
def patch_head_hook(z, hook, head, clean_cache):
    z[:, :, head, :] = clean_cache[hook.name][:, :, head, :]
    return z

patch_imp = np.zeros((n_layers, n_heads))
t0 = time.time()
with torch.no_grad():
    for layer in range(n_layers):
        name = utils.get_act_name("z", layer)
        for head in range(n_heads):
            hook_fn = functools.partial(patch_head_hook, head=head, clean_cache=clean_cache)
            patched = model.run_with_hooks(corrupted_tokens, fwd_hooks=[(name, hook_fn)])
            patch_imp[layer, head] = (logit_diff(patched) - CORRUPT_LD) / denom
patch_time = time.time() - t0

# --- Attribution patching (estimate) ---
filter_z = lambda nm: nm.endswith("hook_z")
corrupt_acts, corrupt_grads = {}, {}
model.reset_hooks()
for l in range(n_layers):
    nm = utils.get_act_name("z", l)
    model.add_hook(nm, lambda a, hook: corrupt_acts.__setitem__(hook.name, a.detach()), "fwd")
    model.add_hook(nm, lambda g, hook: corrupt_grads.__setitem__(hook.name, g.detach()), "bwd")
torch.set_grad_enabled(True)
t0 = time.time()
logit_diff_tensor(model(corrupted_tokens)).backward()
attr_time = time.time() - t0
model.reset_hooks(); torch.set_grad_enabled(False)

attr_imp = np.zeros((n_layers, n_heads))
for l in range(n_layers):
    nm = utils.get_act_name("z", l)
    contrib = (clean_cache[nm] - corrupt_acts[nm]) * corrupt_grads[nm]
    attr_imp[l] = (contrib.sum(dim=(0, 1, 3)) / denom).cpu().numpy()

print(f"Activation patching: {patch_time:.2f}s  |  Attribution patching: {attr_time:.3f}s")

## 3. RQ1 — Agreement
Three complementary measures:
1. **Rank correlation (Spearman)** across all 144 heads — do the methods order heads the same way overall? (1 = identical order, 0 = unrelated).
2. **Top-k overlap (Jaccard)** — of the k heads each calls most important, what fraction are shared?
3. **Precision / recall against the known circuit** (Wang et al., 2023) — does each method's top-k land on the heads researchers hand-validated?

> The circuit head list below is transcribed from Wang et al. (2023). **Verify it against the paper yourself before using it in the thesis.**

In [ ]:
# Known IOI circuit heads (layer, head) - from Wang et al. (2023). VERIFY against the paper.
CIRCUIT_HEADS = {
    (9,6),(9,9),(10,0),                          # name movers
    (10,7),(11,10),                              # negative name movers
    (9,0),(9,7),(10,1),(10,2),(10,6),(10,10),(11,2),(11,9),  # backup name movers
    (7,3),(7,9),(8,6),(8,10),                    # S-inhibition
    (5,5),(5,8),(5,9),(6,9),                     # induction
    (0,1),(0,10),(3,0),                          # duplicate token
    (2,2),(4,11),                                # previous token
}

def ranked_heads(matrix):
    order = [(matrix[l, h], l, h) for l in range(n_layers) for h in range(n_heads)]
    order.sort(reverse=True)
    return [(l, h) for _, l, h in order]

patch_rank = ranked_heads(patch_imp)
attr_rank  = ranked_heads(attr_imp)

# 1. Spearman rank correlation across all heads
rho, _ = spearmanr(patch_imp.flatten(), attr_imp.flatten())
print(f"Spearman rank correlation (patching vs attribution): {rho:.3f}\n")

# 2 & 3. Top-k overlap and precision/recall vs the circuit
print(f"{'k':>3}  {'Jaccard':>8}  {'patch P':>8} {'patch R':>8}  {'attr P':>8} {'attr R':>8}")
for k in [5, 10, 15, 20]:
    P, A = set(patch_rank[:k]), set(attr_rank[:k])
    jac = len(P & A) / len(P | A)
    pp, pr = len(P & CIRCUIT_HEADS)/k, len(P & CIRCUIT_HEADS)/len(CIRCUIT_HEADS)
    ap, ar = len(A & CIRCUIT_HEADS)/k, len(A & CIRCUIT_HEADS)/len(CIRCUIT_HEADS)
    print(f"{k:>3}  {jac:>8.2f}  {pp:>8.2f} {pr:>8.2f}  {ap:>8.2f} {ar:>8.2f}")

## 4. RQ2 — Faithfulness curves
The real test. For each method we rank the heads, then switch off (mean-ablate) the top-k and measure the logit difference.

- **Necessity:** ablate the top-k. A faithful method's curve should drop **fast** — the heads it named really were needed.
- **Sufficiency:** ablate everything *except* the top-k. A faithful method's curve should rise **fast** — those heads alone largely do the job.

If the two methods' curves sit on top of each other, the fast method is as faithful as the slow one.

In [ ]:
# Mean-ablation harness (Stage 2)
mean_z = {l: clean_cache[utils.get_act_name('z', l)].mean(0, keepdim=True) for l in range(n_layers)}

def run_with_ablation(heads):
    by_layer = defaultdict(list)
    for l, h in heads:
        by_layer[l].append(h)
    def make_hook(layer, hs):
        def hook(z, hook):
            for h in hs:
                z[:, :, h, :] = mean_z[layer][:, :, h, :]
            return z
        return hook
    fwd = [(utils.get_act_name('z', l), make_hook(l, hs)) for l, hs in by_layer.items()]
    with torch.no_grad():
        return logit_diff(model.run_with_hooks(clean_tokens, fwd_hooks=fwd))

all_heads = [(l, h) for l in range(n_layers) for h in range(n_heads)]
K = 20

def necessity(rank):
    return [run_with_ablation(rank[:k]) for k in range(K + 1)]
def sufficiency(rank):
    return [run_with_ablation([h for h in all_heads if h not in set(rank[:k])]) for k in range(K + 1)]

nec_patch, nec_attr = necessity(patch_rank), necessity(attr_rank)
suf_patch, suf_attr = sufficiency(patch_rank), sufficiency(attr_rank)
print("Curves computed.")

In [ ]:
ks = list(range(K + 1))
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].plot(ks, nec_patch, 'o-', label='activation patching')
ax[0].plot(ks, nec_attr, 's-', label='attribution patching')
ax[0].axhline(CLEAN_LD, ls='--', c='grey', lw=1)
ax[0].set_title('Necessity: ablate top-k heads (lower = more faithful)')
ax[0].set_xlabel('number of top heads ablated (k)'); ax[0].set_ylabel('logit difference'); ax[0].legend()

ax[1].plot(ks, suf_patch, 'o-', label='activation patching')
ax[1].plot(ks, suf_attr, 's-', label='attribution patching')
ax[1].axhline(CLEAN_LD, ls='--', c='grey', lw=1)
ax[1].set_title('Sufficiency: keep only top-k heads (higher = more faithful)')
ax[1].set_xlabel('number of top heads kept (k)'); ax[1].set_ylabel('logit difference'); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Headline numbers
A couple of single numbers to summarise, alongside the curves.

In [ ]:
print(f"Baseline clean logit difference: {CLEAN_LD:+.3f}\n")
print("After ablating each method's top 10 heads (necessity - lower is more faithful):")
print(f"   activation patching -> {nec_patch[10]:+.3f}")
print(f"   attribution patching -> {nec_attr[10]:+.3f}\n")
print("Keeping only each method's top 10 heads (sufficiency - higher is more faithful):")
print(f"   activation patching -> {suf_patch[10]:+.3f}")
print(f"   attribution patching -> {suf_attr[10]:+.3f}\n")
print(f"Efficiency (RQ3): attribution {attr_time:.3f}s vs activation patching {patch_time:.2f}s "
      f"-> {patch_time/attr_time:.0f}x faster")

## What this gives you (your results section, in miniature)
You now have concrete answers to all three research questions:

- **RQ1 (agreement):** a rank correlation, top-k overlap, and precision/recall against the known circuit.
- **RQ2 (faithfulness):** necessity and sufficiency curves comparing the two methods.
- **RQ3 (efficiency):** a measured speed-up.

**Interpretation to write up (in your own words):** note where the methods agree, where they diverge, whether the fast method is as faithful as the slow one, and what that implies for trusting efficient attribution-based transparency tools. Remember the honest caveats — this is one small model, one task, one template, and a modest sample — which belong in your Limitations section.

**Next steps:** (a) increase the number of name pairs for smoother, more reliable numbers; (b) optionally repeat on a second template or GPT-2 medium as a robustness check *only if time allows*; (c) start writing the Methodology and Results chapters — the figures above are your first real results. Save this notebook and its plots to your repository.